In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [2]:
df = spark.read.json("transactions_10k.jsonl")

In [3]:
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



# Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.

In [4]:
from pyspark.sql.functions import window, avg, round as _round, col

gdansk_lowest_avg = (
    df.filter(col("store") == "Gdańsk")
    .groupBy(window("timestamp", "1 hour"))
    .agg(_round(avg("amount"), 2).alias("srednia_PLN"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "srednia_PLN"
    )
    .orderBy("srednia_PLN")
)

gdansk_lowest_avg.limit(1).show(truncate=False)

+-------------------+-------------------+-----------+
|od                 |do                 |srednia_PLN|
+-------------------+-------------------+-----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|395.01     |
+-------------------+-------------------+-----------+



# Policz ile transakcji per kategoria było w oknie 09:00–09:30.

In [5]:
from pyspark.sql.functions import hour, minute, count

category_0930 = (
    df.filter(
        (hour("timestamp") == 9) &
        (minute("timestamp") <= 30)
    )
    .groupBy("category")
    .agg(count("tx_id").alias("liczba_tx"))
)

category_0930.show()

+-----------+---------+
|   category|liczba_tx|
+-----------+---------+
|    książki|      648|
|     odzież|      624|
|elektronika|      632|
|    żywność|      583|
+-----------+---------+



# Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [6]:
from pyspark.sql.functions import desc

peak_15min = (
    df.groupBy(window("timestamp", "15 minutes"))
    .agg(count("tx_id").alias("liczba_tx"))
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx"
    )
    .orderBy(desc("liczba_tx"))
)

peak_15min.show(truncate=False)

+-------------------+-------------------+---------+
|od                 |do                 |liczba_tx|
+-------------------+-------------------+---------+
|2026-04-12 09:15:00|2026-04-12 09:30:00|1234     |
|2026-04-12 09:00:00|2026-04-12 09:15:00|1171     |
|2026-04-12 09:30:00|2026-04-12 09:45:00|1156     |
|2026-04-12 08:45:00|2026-04-12 09:00:00|1139     |
|2026-04-12 09:45:00|2026-04-12 10:00:00|1100     |
|2026-04-12 08:30:00|2026-04-12 08:45:00|899      |
|2026-04-12 10:00:00|2026-04-12 10:15:00|858      |
|2026-04-12 08:15:00|2026-04-12 08:30:00|644      |
|2026-04-12 10:15:00|2026-04-12 10:30:00|582      |
|2026-04-12 08:00:00|2026-04-12 08:15:00|468      |
|2026-04-12 10:30:00|2026-04-12 10:45:00|443      |
|2026-04-12 10:45:00|2026-04-12 11:00:00|306      |
+-------------------+-------------------+---------+

